In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="FDMC510P-MS", library="Triode_MOS_Tube_Transistor")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
 

* FDMC510P-MS P-channel MOSFET (DFN3x3-8)
* Pin order: S1 S2 S3 G D
.SUBCKT FDMC510P-MS S1 S2 S3 G D
* RS1 S1 S 0.2m
* RS2 S2 S 0.2m
* RS3 S3 S 0.2m
* RDRAIN D D_INT 0.5m
* CGS G S 3n
* CGD G D_INT 1n
MMAIN D_INT G S S FDMC510P_PMOS L=1u W=2000u
.MODEL FDMC510P_PMOS PMOS (LEVEL=1 VTO=-1.8 KP=60u RD=0.002 RS=0.002)
* Simplified Debug Model:
MMAIN D G S1 S1 Debug_PMOS
.MODEL Debug_PMOS PMOS (LEVEL=1 VTO=-1.8 KP=60u)
.ENDS FDMC510P-MS



In [14]:
   # Save part model
from python.spice_tools import save_part_model

with open("test_cases/test_3ACharger/spice/charger/FDMC510P_MS.spice.lib", "r") as f:
    model_content = f.read()
save_part_model(
name="FDMC510P-MS",
library="Triode_MOS_Tube_Transistor",
model_content=model_content,
vendor_provided=False,
)

In [ ]:
from pathlib import Path
from python.spice_tools import convert_skidl_module

name = convert_skidl_module(
    input_path=Path("test_cases/test_charger_3A/skidl/modules/ip2312_charger.py"),
    subckt_name="IP2312_CHARGER",
    output_path=Path("test_cases/test_charger_3A/spice/ip2312_charger/dut.py"),
)

name

In [7]:
import json

test_bench_path = "test_cases/test_3ACharger/testbench/charger_testbench.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases]

case_ids

['startup_cc_5v_vin_3a_limit',
 'cv_float_4p2v_regulation_nominal',
 'cc_at_vin_min_4p5v',
 'vin_max_float_no_overcharge_5p5v',
 'reverse_blocking_vin_off_batt_present']

In [8]:
from python.spice_tools.harness_sanity import harness_sanity_check

for idx in range(len(case_ids)):
    harness_path = f"test_cases/test_3ACharger/spice/charger/testcases/{case_ids[idx]}.py"

    result = harness_sanity_check(harness_path=harness_path)
    print(result)


{'ok': True, 'max_abs_voltage': 0.0, 'max_abs_current': 16.84215803172868, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 4.667370094112973, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.958313101137268, 'max_abs_current': 5.416464741941487, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 4.971716393709661, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 1.1052580783714276, 'max_abs_current': 22.105161567428436, 'num_points': 508, 'error': None}


In [10]:
from python.spice_tools.testbench_runner import run_use_case

for idx in range(len(case_ids)):
    harness_path = f"test_cases/test_3ACharger/spice/charger/testcases/{case_ids[idx]}.py"

    reports = run_use_case(
        schema_path=test_bench_path,
        harness_path=harness_path,
        use_case_name=case_ids[idx],
        dut_path="test_cases/test_3ACharger/spice/charger/dut.py",
        dut_module_name="CHARGER_pyspice",
    )

    print(reports)


{'startup_cc_5v_vin_3a_limit': {'total_measurements': 2, 'num_passed': 1, 'all_passed': False, 'measurements': {'ibat_mean_ge_2a': {'assertion': {'value': -6.399679906010647e-07, 'op': '>=', 'limit': 2.0}, 'passed': False}, 'ibat_mean_le_3p1a': {'assertion': {'value': -6.399679906010647e-07, 'op': '<=', 'limit': 3.1}, 'passed': True}}}}
{'cv_float_4p2v_regulation_nominal': {'total_measurements': 3, 'num_passed': 2, 'all_passed': False, 'measurements': {'vbat_mean_ge_4p15v': {'assertion': {'value': 4.149999883805813, 'op': '>=', 'limit': 4.15}, 'passed': False}, 'vbat_mean_le_4p25v': {'assertion': {'value': 4.149999883805813, 'op': '<=', 'limit': 4.25}, 'passed': True}, 'vbat_ripple_le_150mvpp': {'assertion': {'value': 0.0, 'op': '<=', 'limit': 0.15}, 'passed': True}}}}
{'cc_at_vin_min_4p5v': {'total_measurements': 2, 'num_passed': 2, 'all_passed': True, 'measurements': {'ibat_mean_le_3p1a_vin_min': {'assertion': {'value': -6.399679906002751e-07, 'op': '<=', 'limit': 3.1}, 'passed': Tru

In [12]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "python"))
from python.commands.database_tools.library_schematic import LibraryManager
from python.spice_tools.utils import validate_spice_model

mpn = "FDMC510P-MS"
lib = "Triode_MOS_Tube_Transistor"

result = LibraryManager.get_symbol_pinout({
    "library": lib,
    "symbol": mpn,
})

pins = result.get("pins")
print(pins)

res = validate_spice_model(
    model_path="test_cases/test_3ACharger/spice/charger/FDMC510P_MS.spice.lib",
    expected_subckt_name=mpn,
    expected_pinout=pins,
)

[{'number': '1', 'name': 'S', 'type': 'unspecified'}, {'number': '2', 'name': 'S', 'type': 'unspecified'}, {'number': '3', 'name': 'S', 'type': 'unspecified'}, {'number': '4', 'name': 'G', 'type': 'unspecified'}, {'number': '5', 'name': 'D', 'type': 'unspecified'}]


In [13]:
for r in res:
    print(r+'\n\n')